# Análise de Dados - Microsoft Security Incident Prediction (Silver)

Este notebook conecta diretamente à camada Silver no banco de dados Postgres (`silver.microsoft_security_incident`) e traz uma análise visual de suas principais variáveis. Todas as tabelas e gráficos são baseados nessa fonte, removendo qualquer comparação com a camada Raw.

Os gráficos desta análise permitem observar padrões, tendências e eventuais problemas de qualidade de dados.


In [1]:
# Bibliotecas e conexão com o banco
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

DB_USER = os.getenv('PGUSER', 'postgres')
DB_PASS = os.getenv('PGPASSWORD', 'postgres')
DB_HOST = os.getenv('PGHOST', 'localhost')
DB_PORT = int(os.getenv('PGPORT', '5433'))
DB_NAME = os.getenv('PGDATABASE', 'microsoft-security')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

SILVER_SCHEMA = 'silver'
SILVER_TABLE = 'microsoft_security_incident'
FULL_NAME = f'"{SILVER_SCHEMA}"."{SILVER_TABLE}"'

# Consulta
df = pd.read_sql_query(text(f"SELECT * FROM {FULL_NAME}"), engine)
if 'timestamp' in df.columns:
    df['datetime'] = pd.to_datetime(df['timestamp'], errors='coerce')
print(f'Dataset shape: {df.shape}')
df.head()


MemoryError: Unable to allocate 1.13 GiB for an array with shape (16, 9516837) and data type object

## Visão Geral dos Dados
Resumo inicial: dimensão, tipos e estatísticas básicas.


In [ ]:
display(df.head())
df.info()
display(df.describe(include="all", datetime_is_numeric=True).T)


## Distribuição de Variáveis Categóricas
Histogramas e contagens das principais variáveis categóricas.


In [ ]:
cat_cols = [c for c in [
    "category", "incident_grade", "country_code", "state", "city",
    "last_verdict", "os_family", "os_version", "alert_title",
    "detector_id", "org_id"
] if c in df.columns]
for col in cat_cols:
    plt.figure(figsize=(10,4))
    vc = df[col].value_counts().head(20)
    sns.barplot(x=vc.index.astype(str), y=vc.values)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Distribuição de {col}")
    plt.ylabel("Contagem")
    plt.show()


## Distribuição de Variáveis Numéricas
Histogramas, boxplots e temporal dos principais campos numéricos e de data.


In [ ]:
import matplotlib.dates as mdates
num_cols = df.select_dtypes(include=["number"]).columns
for col in num_cols:
    plt.figure(figsize=(10,4))
    sns.histplot(df[col], bins=30, kde=True, color="skyblue")
    plt.title(f"Distribuição de {col}")
    plt.xlabel(col)
    plt.show()
    # Boxplot
    plt.figure(figsize=(10,2))
    sns.boxplot(x=df[col], color="orange")
    plt.title(f"Boxplot de {col}")
    plt.show()
# Se houver datetime
if "datetime" in df.columns:
    plt.figure(figsize=(12,4))
    df_time = df.sort_values("datetime")
    grp = df_time["datetime"].dt.date.value_counts().sort_index()
    plt.plot(grp.index, grp.values)
    plt.title("Contagem de registros por data")
    plt.xlabel("Data")
    plt.ylabel("Nº registros")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## Mapa de Correlação
Correlação entre variáveis numéricas.


In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1)
plt.title("Mapa de Correlação")
plt.show()
